<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Stochastic-Simulations/blob/master/Przyklad_3_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Self-Avoiding Walks
---

### Estimators for Crude Monte Carlo
---

In the Crude Monte Carlo approach, we blindly generate random walks of length $k$ by uniformly choosing one of the $2d$ possible directions at each step, yielding a total sample space of $(2d)^k$ potential paths. The estimator for the total number of self-avoiding walks, $c$, is computed by multiplying the observed success rate by the total path space:

$$c = \frac{N_{success}}{N} (2d)^k$$

To estimate the mean squared end-to-end distance, $\theta = \mathbb{E}[L^2]$, we simply calculate the arithmetic mean of the squared Euclidean distances $L^2$ strictly for the paths that successfully reached length $k$ without any self-intersections.

### Estimators for the Rosenbluth Algorithm
---

The Rosenbluth algorithm employs importance sampling by exclusively allowing steps into unvisited adjacent sites, actively avoiding immediate collisions. To correct for this altered and biased probability measure, each generated path is assigned a statistical weight $W = \prod_{j=1}^k m_j$, where $m_j$ represents the number of valid, unoccupied neighbors available at step $j$. If a walk becomes trapped before reaching its target length, its weight evaluates to $W = 0$. The total number of valid walks $c$ is then estimated as the expected value of these weights across all $N$ trials:

$$c = \frac{1}{N} \sum_{i=1}^N W_i$$

Consequently, the mean squared distance $\theta$ is calculated as a weighted average of the squared distances $L_i^2$, formulated as $\theta = \frac{\sum W_i L_i^2}{\sum W_i}$. This ensures that paths with a higher number of structural choices proportionately influence the final spatial metric.

### Simulations
---

In [14]:
import numpy as np
import pandas as pd
import random

In [15]:
def get_directions(d):
    """Generates all 2d possible step directions for a d-dimensional grid."""
    directions = []
    for i in range(d):
        # Move forward in dimension i
        v_forward = [0] * d
        v_forward[i] = 1
        directions.append(tuple(v_forward))

        # Move backward in dimension i
        v_backward = [0] * d
        v_backward[i] = -1
        directions.append(tuple(v_backward))
    return directions

In [16]:
def run_saw_simulation(d, k, N=100_000):
    """
    Simulates Self-Avoiding Walks using Crude MC and Rosenbluth algorithms.

    Parameters:
    d (int): Number of dimensions
    k (int): Length of the walk
    N (int): Number of simulated paths
    """
    directions = get_directions(d)
    total_walks_possible = (2 * d) ** k

    # ==========================================
    # 1. Crude Monte Carlo
    # ==========================================
    crude_success_count = 0
    crude_sq_distances = []

    for _ in range(N):
        pos = tuple([0] * d)
        visited = {pos}
        success = True

        for step in range(k):
            # Pick a random direction blindly
            step_dir = random.choice(directions)
            pos = tuple(p + dr for p, dr in zip(pos, step_dir))

            if pos in visited:
                success = False
                break # Collision
            visited.add(pos)

        if success:
            crude_success_count += 1
            sq_dist = sum(coord ** 2 for coord in pos)
            crude_sq_distances.append(sq_dist)

    # Crude MC Estimators
    crude_acceptance_rate = crude_success_count / N
    c_crude = crude_acceptance_rate * total_walks_possible
    theta_crude = np.mean(crude_sq_distances) if crude_success_count > 0 else 0.0

    # ==========================================
    # 2. Rosenbluth Algorithm
    # ==========================================
    rosenbluth_weights = np.zeros(N)
    rosenbluth_sq_distances = np.zeros(N)
    rosenbluth_success_count = 0

    for i in range(N):
        pos = tuple([0] * d)
        visited = {pos}
        weight = 1.0

        for step in range(k):
            # Find all VALID (unvisited) neighbors
            valid_neighbors = []
            for step_dir in directions:
                neighbor = tuple(p + dr for p, dr in zip(pos, step_dir))
                if neighbor not in visited:
                    valid_neighbors.append(neighbor)

            m_i = len(valid_neighbors)

            # If trapped, walk dies and weight becomes 0
            if m_i == 0:
                weight = 0.0
                break

            # Update cumulative weight and pick a valid move
            weight *= m_i
            pos = random.choice(valid_neighbors)
            visited.add(pos)

        rosenbluth_weights[i] = weight
        if weight > 0:
            rosenbluth_success_count += 1
            rosenbluth_sq_distances[i] = sum(coord ** 2 for coord in pos)

    # Rosenbluth Estimators
    c_rosenbluth = np.mean(rosenbluth_weights)

    # Theta is the weighted average of squared distances
    sum_weights = np.sum(rosenbluth_weights)
    if sum_weights > 0:
        theta_rosenbluth = np.sum(rosenbluth_weights * rosenbluth_sq_distances) / sum_weights
    else:
        theta_rosenbluth = 0.0

    rosenbluth_success_rate = rosenbluth_success_count / N

    return {
        "k (Length)": k,
        "c (Crude MC)": c_crude,
        "c (Rosenbluth)": c_rosenbluth,
        "θ (Crude MC)": theta_crude,
        "θ (Rosenbluth)": theta_rosenbluth,
        "Crude Success Rate": f"{crude_acceptance_rate:.2%}",
        "Rosenbluth Success Rate": f"{rosenbluth_success_rate:.2%}"
    }

In [17]:
print("Running simulations for d=2 (2D Lattice)")
results = []
# Testing for walk lengths of 15 and 30
for walk_length in [15, 30]:
    res = run_saw_simulation(d=2, k=walk_length, N=50_000)
    results.append(res)

df_results = pd.DataFrame(results)

# Formatting the output table for better readability
formatted_df = df_results.style.format({
    "c (Crude MC)": "{:.2e}",
    "c (Rosenbluth)": "{:.2e}",
    "θ (Crude MC)": "{:.2f}",
    "θ (Rosenbluth)": "{:.2f}"
}).set_properties(**{'text-align': 'center'})

display(formatted_df)

Running simulations for d=2 (2D Lattice)


,k (Length),c (Crude MC),c (Rosenbluth),θ (Crude MC),θ (Rosenbluth),Crude Success Rate,Rosenbluth Success Rate
0,15,6.68e+06,6.44e+06,50.38,47.42,0.62%,95.74%
1,30,0.00e+00,1.65e+13,0.00,129.41,0.00%,80.19%


In [18]:
print("Running simulations for d=3 (3D Lattice)")
results = []
# Testing for walk lengths of 15 and 30
for walk_length in [15, 30]:
    res = run_saw_simulation(d=3, k=walk_length, N=50_000)
    results.append(res)

df_results = pd.DataFrame(results)

# Formatting the output table for better readability
formatted_df = df_results.style.format({
    "c (Crude MC)": "{:.2e}",
    "c (Rosenbluth)": "{:.2e}",
    "θ (Crude MC)": "{:.2f}",
    "θ (Rosenbluth)": "{:.2f}"
}).set_properties(**{'text-align': 'center'})

display(formatted_df)

Running simulations for d=3 (3D Lattice)


,k (Length),c (Crude MC),c (Rosenbluth),θ (Crude MC),θ (Rosenbluth),Crude Success Rate,Rosenbluth Success Rate
0,15,2.11e+10,2.12e+10,28.14,27.60,4.49%,100.00%
1,30,2.30e+20,2.70e+20,55.73,63.31,0.10%,99.94%


In [19]:
print("Running simulations for d=10 (10D Lattice)")
results = []
# Testing for walk lengths of 15 and 30
for walk_length in [15, 30]:
    res = run_saw_simulation(d=10, k=walk_length, N=50_000)
    results.append(res)

df_results = pd.DataFrame(results)

# Formatting the output table for better readability
formatted_df = df_results.style.format({
    "c (Crude MC)": "{:.2e}",
    "c (Rosenbluth)": "{:.2e}",
    "θ (Crude MC)": "{:.2f}",
    "θ (Rosenbluth)": "{:.2f}"
}).set_properties(**{'text-align': 'center'})

display(formatted_df)

Running simulations for d=10 (10D Lattice)


,k (Length),c (Crude MC),c (Rosenbluth),θ (Crude MC),θ (Rosenbluth),Crude Success Rate,Rosenbluth Success Rate
0,15,1.53e+19,1.54e+19,16.80,16.73,46.79%,100.00%
1,30,2.25e+38,2.23e+38,33.57,33.69,20.97%,100.00%


In [ ]:
print("Running simulations for d=100 (100D Lattice)")
results = []
# Testing for walk lengths of 15 and 30
for walk_length in [15, 30]:
    res = run_saw_simulation(d=100, k=walk_length, N=50_000)
    results.append(res)

df_results = pd.DataFrame(results)

# Formatting the output table for better readability
formatted_df = df_results.style.format({
    "c (Crude MC)": "{:.2e}",
    "c (Rosenbluth)": "{:.2e}",
    "θ (Crude MC)": "{:.2f}",
    "θ (Rosenbluth)": "{:.2f}"
}).set_properties(**{'text-align': 'center'})

display(formatted_df)

Running simulations for d=100 (100D Lattice)
